### **1. Importación de Librerias.**

In [8]:
import pandas as pd
import numpy as np

### **2. Carga del archivo.**

In [9]:
try:
    print("Leyendo archivo: tr_endutih_usuarios2_anual_2022_mod.xlsx")
    df = pd.read_excel("tr_endutih_usuarios2_anual_2022_mod.xlsx", sheet_name=0, engine='openpyxl')
except FileNotFoundError:
    print(f"ERROR: No se encontró tr_endutih_usuarios2_anual_2022_mod.xlsx. Verifica el nombre.")

Leyendo archivo: tr_endutih_usuarios2_anual_2022_mod.xlsx


### **3. Establecer las columnas de interés**

- EDAD: Edad del elegido, (numérica) [0-99].
- P8_4_2: ¿El celular que usa es… inteligente (Smartphone)?, (1:sí, 2:no) [1-2].
- SEXO: Sexo del elegido, (1:hombre, 2:mujer).
- ENT: Estado de la republica, solo se usara para analizar estos estados.

In [10]:
try:
    cols = ["EDAD", "P8_4_2", "SEXO", "ENT"]
except KeyError as e:
    print(f"ERROR: Columna no encontrada: {e}")
    print(f"Columnas disponibles: {list(df.columns)}")

### **Pregunta que queremos, responder: ¿Existe relación entre la edad y el tipo de celular que usan los mexicanos?**

y utilizando la variable de SEXO, queremos responder:

### **¿La asociación entre edad y tipo de celular difiere entre hombres y mujeres?**

y la pregunta de regresión logistica

### **“¿Qué factores (edad y sexo) están asociados con la probabilidad de usar un smartphone, y existe una interacción entre ellos?”**

### **4. Limpieza de nulos y verificación del tamaño.**

In [11]:
# Eliminar filas con nulos en estas columnas
df_clean = df.dropna(subset=cols)

n_limpios = len(df_clean)
print(f"Registros completos (sin nulos en las 3 variables): {n_limpios}")

# Verificar que haya al menos 300 registros para la muestra
if n_limpios < 300:
    print(f"ERROR: Solo hay {n_limpios} registros. Se necesitan al menos 300 para el muestreo.")
    exit()

Registros completos (sin nulos en las 3 variables): 48988


### **4.b. Filtrar por estados seleccionados.**

- ENT[7] = Chiapas.
- ENT[12] = Guerrero.
- ENT[20] = Oaxaca.

In [12]:
# Filtrar solo los estados de interés (7, 12, 20)
estados_interes = [7, 12, 20]
df_filtrado = df_clean[df_clean['ENT'].isin(estados_interes)].copy()
print(f"Registros en estados {estados_interes}: {len(df_filtrado)}")

# Muestreo estratificado: hasta 100 por estado, indicando cuantos hay y cuantos se usaron.
muestras_estado = []
for estado in estados_interes:
    df_estado = df_filtrado[df_filtrado['ENT'] == estado]
    disponibles = len(df_estado)
    n_muestra = min(100, disponibles)
    if disponibles < 100:
        print(f"Estado {estado}: solo {disponibles} registros, se toman todos.")
    else:
        print(f"Estado {estado}: {disponibles} disponibles → 100 seleccionados.")
    muestra_estado = df_estado.sample(n=n_muestra, random_state=42)
    muestras_estado.append(muestra_estado)

Registros en estados [7, 12, 20]: 4050
Estado 7: 1294 disponibles → 100 seleccionados.
Estado 12: 1425 disponibles → 100 seleccionados.
Estado 20: 1331 disponibles → 100 seleccionados.


### **5. Obtención de Submuestras.**

In [13]:
# Unir y eliminar la columna ENT
muestra_300 = pd.concat(muestras_estado, ignore_index=True)
muestra_300 = muestra_300[["EDAD", "P8_4_2", "SEXO"]]   # Solo estas columnas

# Dividir en 3 submuestras equitativas (tamaño variable)
rng = np.random.RandomState(42)
indices_perm = rng.permutation(len(muestra_300))
n_total = len(muestra_300)
tam_base = n_total // 3
resto = n_total % 3
tams = [tam_base + (1 if i < resto else 0) for i in range(3)]

# Guardar las submuestras
sub1 = muestra_300.iloc[indices_perm[:tams[0]]].reset_index(drop=True)
sub2 = muestra_300.iloc[indices_perm[tams[0]:tams[0]+tams[1]]].reset_index(drop=True)
sub3 = muestra_300.iloc[indices_perm[tams[0]+tams[1]:]].reset_index(drop=True)

# Impresion del guardado exitoso
print(f"Muestra combinada (estratificada): {n_total} registros")
print(f"Tamaños de submuestras: {len(sub1)}, {len(sub2)}, {len(sub3)}")

Muestra combinada (estratificada): 300 registros
Tamaños de submuestras: 100, 100, 100


### **6. Guardar las submuestras.**

In [14]:
sub1.to_csv("submuestra1.csv", index=False)
sub2.to_csv("submuestra2.csv", index=False)
sub3.to_csv("submuestra3.csv", index=False)

# Impresion de que se completo el guardado
print(f"Registros originales en la hoja: {len(df)}")
print(f"Se extrajo una muestra de 300 y se partió en 3 submuestras de 100.")
print(f"Archivos generados: submuestra1.csv, submuestra2.csv, submuestra3.csv")

Registros originales en la hoja: 58540
Se extrajo una muestra de 300 y se partió en 3 submuestras de 100.
Archivos generados: submuestra1.csv, submuestra2.csv, submuestra3.csv


### **7. Dataframe combinado con identificador (para el análisis global)**

In [15]:
# Unir las submuestras indicando de que submuestra provienen
df_total = pd.concat(
    [sub1.assign(subconjunto="Submuestra 1"),
        sub2.assign(subconjunto="Submuestra 2"),
        sub3.assign(subconjunto="Submuestra 3")],
    ignore_index=True
)

### **8. Filtrar por sexo el dataframe total con todas sus columnas**

In [16]:
df_hombres = df_total[df_total['SEXO'] == 1]
df_mujeres = df_total[df_total['SEXO'] == 2]

# Imprimir la distribucion del sexo
print(f"Tamaño total: {len(df_total)}")
print(f"Hombres: {len(df_hombres)} | Mujeres: {len(df_mujeres)}")

Tamaño total: 300
Hombres: 131 | Mujeres: 169


### **9. Guardar datasets de hombres y mujeres.**

In [17]:
df_hombres.to_csv("sub_hombres.csv", index=False)
df_mujeres.to_csv("sub_mujeres.csv", index=False)